In [ ]:

import pandas as pd

# Read the first CSV file into a DataFrame
df_irs_repairs = pd.read_csv("../../datasets/irs_repairs.csv")

df_irs_repairs

In [ ]:
len(df_irs_repairs[(df_irs_repairs['Constraint Deleted'] == True)] )

In [ ]:
len(df_irs_repairs[(df_irs_repairs['Constraint Deprecated'] == True)] )

In [ ]:
len(df_irs_repairs[(df_irs_repairs['Included as Exception'] == True)] )

In [ ]:
df_irs_repairs['A-box wdt statement Deleted'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def statementDeleted(row):
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{wdt_pid}> [] }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
print(statementDeleted(df_irs_repairs.iloc[2]))

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_irs_repairs.iterrows(), total=len(df_irs_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['A-box wdt statement Deleted']):
        result = statementDeleted(row)
        df_irs_repairs.at[index, 'A-box wdt statement Deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 300000 == 0 and index != 0:
        df_irs_repairs.to_csv("checkpoint_irs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_irs_repairs.to_csv("checkpoint_irs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_irs_repairs.to_csv("checkpoint_irs.csv", index=False)

In [ ]:
df_irs_repairs

In [ ]:
len(df_irs_repairs[(df_irs_repairs['A-box wdt statement Deleted'] == True)] )

In [ ]:
df_irs_repairs['A-box wdt required statement added'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def onlyReqPropStmtAdded(row):
    
    if row['2019_no_req_val'] is False:
        return None
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{row['wdt_required_property']}> []}}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
print(onlyReqPropStmtAdded(df_irs_repairs.iloc[0]))

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getRequiredValues(row, endpoint = "ENTER_qEndpoint_WD_2019"):
        
    if row['2019_no_req_val'] is True:
        return None
    
    wd_required_pid = row['wdt_required_property'].replace("http://www.wikidata.org/prop/direct/", "http://www.wikidata.org/entity/")
    
    # SPARQL query
    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>

        SELECT 
          ?value
        WHERE
        {{
          <{row['property']}> p:P2302 ?statement.
          ?statement ps:P2302 wd:Q21503247. ## irs
          ?statement pq:P2306 <{wd_required_pid}>.
          ?statement pq:P2305 ?value. 
        }}
    """
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find all 'uri' elements and extract their text
        uris = [uri_element.text for uri_element in root.findall('.//ns:uri', namespace)]

        if len(uris) == 0:
            return []
        
        return uris
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
    
def reqPropValStmtAdded(row, required_value):
    
    if row['2019_no_req_val'] is True:
        return None
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{row['wdt_required_property']}> <{required_value}>}}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
def testReqStmtWithReqValueAdded(row):
    reqValues = getRequiredValues(row)
    #print(reqValues)
    if reqValues is None:
        return None
    if len(reqValues) == 0:
        return None
    for reqVal in reqValues:
        result = reqPropValStmtAdded(row, reqVal)
        if result is True:
            return True
    return False

testReqStmtWithReqValueAdded(df_irs_repairs.iloc[2578219])

In [ ]:
df_irs_repairs[(df_irs_repairs['2019_no_req_val'] == False)]

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_irs_repairs.iterrows(), total=len(df_irs_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['A-box wdt required statement added']):
        if row['2019_no_req_val'] is True:
            result = onlyReqPropStmtAdded(row)
            df_irs_repairs.at[index, 'A-box wdt required statement added'] = result
        else:
            result = testReqStmtWithReqValueAdded(row)
            df_irs_repairs.at[index, 'A-box wdt required statement added'] = result

    # Save a checkpoint every 10,000 rows
    if index % 500000 == 0 and index != 0:
        df_irs_repairs.to_csv("checkpoint_irs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output 
df_irs_repairs.to_csv("checkpoint_irs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_irs_repairs

In [ ]:
len(df_irs_repairs[(df_irs_repairs['A-box wdt statement Deleted'] == True)] )

In [ ]:
len(df_irs_repairs[(df_irs_repairs['A-box wdt required statement added'] == True)] )

In [ ]:
df_irs_repairs['t-box required property changed'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getConstraintID(row, endpoint = "ENTER_qEndpoint_WD_2019"):
        
    #wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    #endpoint = "https://wikidata.ai.wu.ac.at/wd2019"

    if bool(row['2019_no_req_val']):
        req_val = "FILTER NOT EXISTS {?statement pq:P2305 []}"
    else:
        req_val = "?statement pq:P2305 []."
    
    # SPARQL query
    query = f"""
            PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
            PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX wikibase: <http://wikiba.se/ontology#>
            PREFIX p: <http://www.wikidata.org/prop/>
            PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
            PREFIX ps: <http://www.wikidata.org/prop/statement/>
            PREFIX wd: <http://www.wikidata.org/entity/>
            PREFIX wdt: <http://www.wikidata.org/prop/direct/>

            SELECT ?statement {{
              ?statement ps:P2302 wd:Q21503247. ## item-requires-statement constraint
              <{row['property']}> p:P2302 ?statement.
              ?statement pq:P2306/wikibase:directClaim <{row['wdt_required_property']}>.
              # exception and deprecation
              FILTER NOT EXISTS {{?statement pq:P2241 []}}
              FILTER NOT EXISTS {{?statement wikibase:rank wikibase:DeprecatedRank}}
              {req_val}
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        #print(response.text)

        # Parse the XML data
        namespace = {'sparql': 'http://www.w3.org/2005/sparql-results#'}
        root = ET.fromstring(response.text)

        # Find the uri inside the binding
        if root.find('.//sparql:binding[@name="statement"]/sparql:uri', namespace) is not None:
            return root.find('.//sparql:binding[@name="statement"]/sparql:uri', namespace).text
        return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
    
def getRequiredProperty(constraint_statement):
        
    #wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"
    
    # SPARQL query
    query = f"""
            PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
            PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX wikibase: <http://wikiba.se/ontology#>
            PREFIX p: <http://www.wikidata.org/prop/>
            PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
            PREFIX ps: <http://www.wikidata.org/prop/statement/>
            PREFIX wd: <http://www.wikidata.org/entity/>
            PREFIX wdt: <http://www.wikidata.org/prop/direct/>

            SELECT ?wdt_required_property {{
              <{constraint_statement}> pq:P2306/wikibase:directClaim ?wdt_required_property.
              # exception and deprecation
              FILTER NOT EXISTS {{<{constraint_statement}> pq:P2241 []}}
              FILTER NOT EXISTS {{<{constraint_statement}> wikibase:rank wikibase:DeprecatedRank}}
            }}
            """
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        #print(response.text)

        # Parse the XML data
        namespace = {'sparql': 'http://www.w3.org/2005/sparql-results#'}
        root = ET.fromstring(response.text)

        # Find the uri inside the binding
        if root.find('.//sparql:binding[@name="wdt_required_property"]/sparql:uri', namespace) is not None:
            return root.find('.//sparql:binding[@name="wdt_required_property"]/sparql:uri', namespace).text
        return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
def hasRequiredPropertyChanged(row):
    constraint_id_2019 = getConstraintID(row)
    if constraint_id_2019 is None:
        return None
    req_prop_2023 = getRequiredProperty(constraint_id_2019)
    if req_prop_2023 is None:
        return False
    #print("old :", row['wdt_required_property'])
    #print("new :", req_prop_2023)
    if req_prop_2023 != row['wdt_required_property']:
        return True
    return False
        
    
# Example usage 
hasRequiredPropertyChanged(df_irs_repairs.iloc[3660701])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_irs_repairs.iterrows(), total=len(df_irs_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['t-box required property changed']):
        result = hasRequiredPropertyChanged(row)
        df_irs_repairs.at[index, 't-box required property changed'] = result

    # Save a checkpoint every 10,000 rows
    if index % 1000000 == 0 and index != 0:
        df_irs_repairs.to_csv("checkpoint_irs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_irs_repairs.to_csv("checkpoint_irs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_irs_repairs[(df_irs_repairs['t-box required property changed'] == True)] )

In [ ]:
df_irs_repairs['t-box required value changed'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getCountRequiredValues(row, endpoint):

    if bool(row['2019_no_req_val']) and endpoint == "ENTER_qEndpoint_WD_2019":
        return 0
    
    
    # SPARQL query
    query = f"""
            PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
            PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX wikibase: <http://wikiba.se/ontology#>
            PREFIX p: <http://www.wikidata.org/prop/>
            PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
            PREFIX ps: <http://www.wikidata.org/prop/statement/>
            PREFIX wd: <http://www.wikidata.org/entity/>
            PREFIX wdt: <http://www.wikidata.org/prop/direct/>

            SELECT (COUNT(?expected_values) as ?total) {{
              ?statement ps:P2302  wd:Q21503247. ## item-requires-statement constraint
              <{row['property']}> p:P2302 ?statement.
              ?statement pq:P2306/wikibase:directClaim <{row['wdt_required_property']}>.
              # exception and deprecation
              FILTER NOT EXISTS {{?statement pq:P2241 []}}
              FILTER NOT EXISTS {{?statement wikibase:rank wikibase:DeprecatedRank}}
              ?statement pq:P2305 ?expected_values.
            }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:

        # Parse the XML data
        namespace = {'sparql': 'http://www.w3.org/2005/sparql-results#'}
        root = ET.fromstring(response.text)

        # Find the uri inside the binding
        if root.find('.//sparql:binding[@name="total"]/sparql:literal', namespace) is not None:
            return root.find('.//sparql:binding[@name="total"]/sparql:literal', namespace).text
        return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
def hasRequiredValuesNumberIncreased(row):
    n_2019 = getCountRequiredValues(row,"ENTER_qEndpoint_WD_2019")
    if n_2019 is None:
        return None
    n_2023 = getCountRequiredValues(row,"ENTER_qEndpoint_WD_2023")
    if n_2023 is None:
        return None
    if int(n_2023) > int(n_2019):
        return True
    return False
    
print(getCountRequiredValues(df_irs_repairs.iloc[3669517],"ENTER_qEndpoint_WD_2019"))
print(getCountRequiredValues(df_irs_repairs.iloc[3669517],"ENTER_qEndpoint_WD_2023"))
hasRequiredValuesNumberIncreased(df_irs_repairs.iloc[3669517])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_irs_repairs.iterrows(), total=len(df_irs_repairs)):
    
    if pd.isna(row['t-box required value changed']) or row['t-box required value changed'] is None:
        if bool(row['Constraint Deleted']) is True:
            df_irs_repairs.at[index, 't-box required value changed'] = False
        else:
            result = hasRequiredValuesNumberIncreased(row)
            df_irs_repairs.at[index, 't-box required value changed'] = result

    # Save a checkpoint every 10,000 rows
    if index % 1000000 == 0 and index != 0:
        df_irs_repairs.to_csv("checkpoint_irs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_irs_repairs.to_csv("checkpoint_irs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_irs_repairs[(df_irs_repairs['t-box required value changed'] == True)] )

In [ ]:
df_irs_repairs

In [ ]:
df_irs_repairs[
    (df_irs_repairs['Constraint Deleted'] == False)
    & (df_irs_repairs['Constraint Deprecated'] == False)
    & (df_irs_repairs['Included as Exception'] == False)
    & (df_irs_repairs['A-box wdt statement Deleted'] == False)
    & (df_irs_repairs['t-box required value changed'] == False)
    & (df_irs_repairs['t-box required property changed'] == False)
    & ((df_irs_repairs['A-box wdt required statement added'] == False) 
       | (pd.isna(df_irs_repairs['A-box wdt required statement added'])))
]

In [ ]:
df_irs_repairs['t-box required value removed'] = None

In [ ]:
getCountRequiredValues(df_irs_repairs.iloc[2632021], endpoint='ENTER_qEndpoint_WD_2019')

In [ ]:
getCountRequiredValues(df_irs_repairs.iloc[2632021], endpoint='ENTER_qEndpoint_WD_2023')

In [ ]:
def requiredValueRemoved(row):
    n_2019 = getCountRequiredValues(row,"ENTER_qEndpoint_WD_2019")
    if n_2019 is None:
        return None
    n_2023 = getCountRequiredValues(row,"ENTER_qEndpoint_WD_2023")
    if n_2023 is None:
        return None
    if int(n_2023) == 0 and int(n_2019) > 0:
        return True
    return False

In [ ]:
requiredValueRemoved(df_irs_repairs.iloc[2632021])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_irs_repairs.iterrows(), total=len(df_irs_repairs)):
    
    if pd.isna(row['t-box required value removed']) or row['t-box required value removed'] is None:
        if bool(row['Constraint Deleted']) is True:
            df_irs_repairs.at[index, 't-box required value removed'] = False
        else:
            result = requiredValueRemoved(row)
            df_irs_repairs.at[index, 't-box required value removed'] = result

    # Save a checkpoint every 10,000 rows
    if index % 1000000 == 0 and index != 0:
        df_irs_repairs.to_csv("checkpoint_irs.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_irs_repairs.to_csv("checkpoint_irs.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_irs_repairs['t-box required value removed'].value_counts()

In [ ]:
df_irs_repairs[
    (df_irs_repairs['Constraint Deleted'] == False)
    & (df_irs_repairs['Constraint Deprecated'] == False)
    & (df_irs_repairs['Included as Exception'] == False)
    & (df_irs_repairs['A-box wdt statement Deleted'] == False)
    & (df_irs_repairs['t-box required value changed'] == False)
    & (df_irs_repairs['t-box required value removed'] == False)
    & (df_irs_repairs['t-box required property changed'] == False)
    & ((df_irs_repairs['A-box wdt required statement added'] == False) 
       | (pd.isna(df_irs_repairs['A-box wdt required statement added'])))
]

In [ ]:
df_unknown = df_irs_repairs[
    (df_irs_repairs['Constraint Deleted'] == False)
    & (df_irs_repairs['Constraint Deprecated'] == False)
    & (df_irs_repairs['Included as Exception'] == False)
    & (df_irs_repairs['A-box wdt statement Deleted'] == False)
    & (df_irs_repairs['t-box required value changed'] == False)
    & (df_irs_repairs['t-box required value removed'] == False)
    & (df_irs_repairs['t-box required property changed'] == False)
    & ((df_irs_repairs['A-box wdt required statement added'] == False) 
       | (pd.isna(df_irs_repairs['A-box wdt required statement added'])))
]

In [ ]:
df_unknown.iloc[0]

In [ ]:
df_unknown['2019_no_req_val'].value_counts()

In [ ]:
getConstraintID(df_unknown.iloc[0])

In [ ]:
getRequiredValues(df_unknown.iloc[0])

In [ ]:
getRequiredValues(df_unknown.iloc[0], endpoint = "ENTER_qEndpoint_WD_2023")

In [ ]:
getRequiredValues(df_irs_repairs.iloc[0], endpoint = "ENTER_qEndpoint_WD_2023")

In [ ]:
import requests
import xml.etree.ElementTree as ET

def stmtAddedForBlankNodes(row):
    
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    filter_condition = ''
    
    req_values = getRequiredValues(row, endpoint = "ENTER_qEndpoint_WD_2023")
    if len(req_values) > 0:
        formatted_values = ', '.join(f'<{value}>' for value in req_values)
        filter_condition = f'FILTER (?o IN ({formatted_values}))'
    
    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{row['wdt_required_property']}> ?o.
        {filter_condition}
    }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
print(stmtAddedForBlankNodes(df_irs_repairs.iloc[0]))
print(stmtAddedForBlankNodes(df_unknown.iloc[0]))

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_unknown.iterrows(), total=len(df_unknown)):
    
    result = stmtAddedForBlankNodes(row)
    df_unknown.at[index, 'A-box wdt required statement added'] = result


# Save the final output
df_unknown.to_csv("df_unknown.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_unknown['A-box wdt required statement added'].value_counts()

In [ ]:
df_unknown

In [ ]:
df_irs_repairs[
    (df_irs_repairs['Constraint Deleted'] == False)
    & (df_irs_repairs['Constraint Deprecated'] == False)
    & (df_irs_repairs['Included as Exception'] == False)
    & (df_irs_repairs['A-box wdt statement Deleted'] == False)
    & (df_irs_repairs['t-box required value changed'] == False)
    & (df_irs_repairs['t-box required value removed'] == False)
    & (df_irs_repairs['t-box required property changed'] == False)
    & ((df_irs_repairs['A-box wdt required statement added'] == False) 
       | (pd.isna(df_irs_repairs['A-box wdt required statement added'])))
]

In [ ]:
df_irs_repairs_copy = df_irs_repairs

In [ ]:
# Define the conditions
conditions = (
    (df_irs_repairs['Constraint Deleted'] == False) &
    (df_irs_repairs['Constraint Deprecated'] == False) &
    (df_irs_repairs['Included as Exception'] == False) &
    (df_irs_repairs['A-box wdt statement Deleted'] == False) &
    (df_irs_repairs['t-box required value changed'] == False) &
    (df_irs_repairs['t-box required value removed'] == False) &
    (df_irs_repairs['t-box required property changed'] == False) &
    ((df_irs_repairs['A-box wdt required statement added'] == False) | 
     (pd.isna(df_irs_repairs['A-box wdt required statement added'])))
)

# Apply the condition and set the value
# Because all unknown rows were true
df_irs_repairs.loc[conditions, 'A-box wdt required statement added'] = True

In [ ]:
df_irs_repairs.iloc[2633919]

In [ ]:
df_irs_repairs['A-box wdt required statement added'].value_counts()

In [ ]:
df_irs_repairs[
    (df_irs_repairs['Constraint Deleted'] == False)
    & (df_irs_repairs['Constraint Deprecated'] == False)
    & (df_irs_repairs['Included as Exception'] == False)
    & (df_irs_repairs['A-box wdt statement Deleted'] == False)
    & (df_irs_repairs['t-box required value changed'] == False)
    & (df_irs_repairs['t-box required value removed'] == False)
    & (df_irs_repairs['t-box required property changed'] == False)
    & ((df_irs_repairs['A-box wdt required statement added'] == False) 
       | (pd.isna(df_irs_repairs['A-box wdt required statement added'])))
]

In [ ]:
df_irs_repairs.to_csv("irs_all_repairs.csv", index=False)

In [ ]:
df_irs_repairs['A-box wdt statement Deleted'].value_counts()